In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

False

In [6]:
MODEL="llama3.2"

In [13]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaEmbeddings

llm = ChatOllama(model=MODEL)
embeddings=OllamaEmbeddings(model="nomic-embed-text")

parser = StrOutputParser()
chain= llm | parser
chain.invoke("Give a fancy word of the day and its meaning")

'Today\'s Fancy Word of the Day is:\n\n**Garrulous**\n\nMeaning: Talkative or loquacious, often to an excessive degree. A person who is excessively talkative can be seen as garrulous.\n\nExample sentence: "My uncle was notorious for being garrulous during dinner parties, dominating the conversation with his lengthy stories."\n\nEtymology: From Latin "garrulus," meaning "peacock" or "talkative," likely due to the bird\'s tendency to display its plumage by making loud calls.\n\nNow, go impress your friends with your newfound vocabulary!'

In [14]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("WEEK-1.pdf")

pages=loader.load_and_split()

In [15]:
from langchain_core.prompts import PromptTemplate
template = """
Answer the question based on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""
prompt=PromptTemplate.from_template(template)
prompt.format(context="Here is some context",question="Here is the question")

'\nAnswer the question based on the context below. If you can\'t \nanswer the question, reply "I don\'t know".\n\nContext: Here is some context\n\nQuestion: Here is the question\n'

In [16]:
chain =  prompt | llm | parser

In [17]:
chain.invoke(
    {
        "context":"Today is Tuesday at 9 PM",
        "question":"What is the time today?"
    }
)

'The current time is 9 PM.'

In [19]:
from langchain_community.vectorstores import DocArrayInMemorySearch
vectorstore=DocArrayInMemorySearch.from_documents(
    pages,
    embedding=embeddings
)

In [20]:
retriever= vectorstore.as_retriever()
retriever.invoke("Ethical Hacking")

[Document(metadata={'producer': '3-Heights™ PDF Toolbox API 6.12.0.6 (http://www.pdf-tools.com)', 'creator': 'PowerPoint', 'creationdate': '2019-07-22T15:42:08+00:00', 'moddate': '2022-01-17T06:45:46+00:00', 'source': 'WEEK-1.pdf', 'total_pages': 79, 'page': 1, 'page_label': '2'}, page_content='q\u202f\tWhat\tis\tethical\thacking?\t\nq\u202f\tPenetra1on\ttes1ng\t\nq\u202f\tRole\tof\tthe\tethical\thacker'),
 Document(metadata={'producer': '3-Heights™ PDF Toolbox API 6.12.0.6 (http://www.pdf-tools.com)', 'creator': 'PowerPoint', 'creationdate': '2019-07-22T15:42:08+00:00', 'moddate': '2022-01-17T06:45:46+00:00', 'source': 'WEEK-1.pdf', 'total_pages': 79, 'page': 2, 'page_label': '3'}, page_content="What\tis\tEthical\tHacking?\t\n•\u202fIt\trefers\tto\tthe\tact\tof\tloca1ng\tweaknesses\tand\tvulnerabili1es\tof\tcomputer\tand\t\ninforma1on\tsystems\tby\treplica1ng\tthe\tintent\tand\tac1ons\tof\tmalicious\thackers.\t\n•\u202fIt\tis\talso\tknown\tas\tpenetra'on\ttes'ng,\tintrusion\ttes'ng\to

In [21]:
from operator import itemgetter
chain=(
    {
        "context":itemgetter("question") | retriever,
        "question":itemgetter("question")
    }
    | prompt
    | llm
    | parser
)
chain.invoke({"question":"what is ethical hacking?"})

"What is Ethical Hacking?\n\nEthical hacking, also known as penetration testing or white-hat hacking, refers to the act of locating weaknesses and vulnerabilities in computer systems and information systems by replicating the intent and actions of malicious hackers. It is also known as penetration testing, intrusion testing, or red teaming.\n\nIt involves employing ethical hackers by companies to perform penetration tests. The goals of these tests include:\n\n1. Penetration test\n2. Legal Acceptance Test\n3. Security test\n\nThe purpose of security testing is to identify vulnerabilities and weaknesses in the system's security policy and procedures. However, unlike malicious hacking, where the goal is to break into a network and cause harm, ethical hacking aims to provide solutions to secure or protect the network.\n\nEthical hackers have knowledge of network and computer technology, are able to communicate with management and IT personnel, understand the laws, and are skilled in using 

In [23]:
questions=[
    "How many bits are used to represent a MAC address",
    "which OSI Layer is responsible for framing,MAC addressing and error detection over a physical link?"
]
for question in questions:
    print(f"Question: {question}")
    print(chain.invoke({'question':question}))
    print()

Question: How many bits are used to represent a MAC address
Based on the provided context, it appears that a MAC (Media Access Control) address is represented by 48 bits. This information can be found in the page content of document metadata with 'page': 57 and 'page_label': '58', where it says "Physical Address (48 bits)".

Question: which OSI Layer is responsible for framing,MAC addressing and error detection over a physical link?
Based on the provided context, the OSI layer responsible for framing, MAC addressing, and error detection over a physical link is the Data Link layer (Layer 2).



In [25]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("WEEK-3.pdf")

page=loader.load_and_split()

questions=[
    "Which routing table flag  indicates that a route is specific for a single host rather than an entire network?",
    "In the OSFP routing protocol,which type of packet is used to determine whether a neighbouring router is active?",
    "What type of address should be used when a packet needs t be delivered to every host within a specific network?",
    "What is the address used to represent the default route in routing table in IPv4?",
    "How many bits are used for hop limit in IPv6 base header?"
]
for question in questions:
    print(f"Question: {question}")
    print(chain.invoke({'question':question}))
    print()

Question: Which routing table flag  indicates that a route is specific for a single host rather than an entire network?
The answer to the question can be inferred from the provided page content. In the section "IP Header Fields (contd.)", the flag is mentioned as "Time to Live (8 bits)". According to this, the Time to Live field identifies the number of packets a router should give up after routing this packet before receiving a response back. This value is usually set by the sender and is decremented each time the packet is forwarded.

Given that the question asks for the flag indicating a route is specific for a single host rather than an entire network, we can infer that the "Time to Live" field serves this purpose. It sets a limit on how long the packet should be alive in the network before it is discarded, which prevents packets from traveling in loops.

This means that routers use the Time to Live (TTL) field to specify the scope of a route - whether it applies to an entire netwo